### Librerías a utilizar

In [63]:
import pandas as pd
import duckdb
import os
import matplotlib.pyplot as plt

Se crea un diccionario con los nombres de archivos para luego cargar las tablas en la base de datos

In [5]:
data_folder = 'Data'

tables = os.listdir(data_folder)

files = {}
for t in tables:
    if t[-3:] == 'csv':
        files[t[:-4]] = t


Se cargan las tablas a duckDB, Por el momento se mantiene la estructura en RAM sin presistencia.

In [ ]:
con = duckdb.connect() # o duckdb.connect('mi_base.duckdb') para persistir

for table, file in files.items():
    con.execute(f"""
        CREATE TABLE {table} AS 
        SELECT * FROM read_csv_auto('Data/{file}')
    """)

# Verificar qué tablas quedaron cargadas
con.execute("SHOW TABLES").fetchdf()


,name
0,activos
1,evento_repuesto
2,evento_tecnico
3,eventos_mantenimiento
4,log_operativo
5,mediciones
6,ordenes
7,plan_mantenimiento_preventivo
8,plantas
9,repuestos_insumos


Explorar tabla **tipo_activo**

In [41]:
con.execute(
    f"""
    SELECT * FROM tipo_activo;
    """
).fetchdf()

,tipo_activo_id,nombre,categoria
0,1,Área/Sector,Estructural
1,2,Bomba centrífuga,Hidráulico/Mecánico
2,3,Bomba hidráulica,Hidráulico
3,4,Motor eléctrico CC,Eléctrico
4,5,Motor eléctrico AC,Eléctrico
5,6,Motor diesel,Combustión
6,7,Motor hidráulico,Hidráulico
7,8,Transformador,Eléctrico
8,9,Generador eléctrico,Eléctrico
9,10,Pistón hidráulico,Hidráulico


In [48]:
con.execute(
    f"""
    SELECT COUNT(*) - COUNT(DISTINCT tipo_activo_id) as duplicated_id FROM tipo_activo;
    """
).fetchdf()

,duplicated_id
0,0


Explorar tabla **activos**

In [53]:
con.execute(
    f"""
    SELECT * FROM activos LIMIT 5
    """
).fetchdf()

,activo_id,padre_id,planta_id,tipo_activo_id,codigo,nivel_jerarquico,posicion_relativa,numero_serie,reemplaza_a,criticidad,fecha_alta,estado,caracteristicas
0,1,NaN,1,1,AREA-1,1,N/A (nodo de área),None,None,Media,2018-03-01,activo,{}
1,2,1.0,1,20,EQ-2,2,fondo del galpón,M-46976,None,Media,2018-03-01,activo,{}
2,3,1.0,1,19,EQ-3,2,Rack 3,PG-81610,None,Alta,2018-03-01,activo,{}
3,4,1.0,1,19,EQ-4,2,PASILLO 2,PG-60804,None,Media,2018-03-01,activo,{}
4,5,1.0,1,20,EQ-5,2,junto a bomba anterior,M-18750,None,Baja,2018-03-01,activo,{}


In [70]:
count_activos = con.execute(
    f"""
    SELECT ta.nombre AS Equipo,
    COUNT(a.activo_id) AS count
    FROM activos a LEFT JOIN tipo_activo ta ON a.tipo_activo_id = ta.tipo_activo_id
    GROUP BY ta.nombre
    ORDER BY COUNT(a.activo_id) DESC; 
    """
).fetchdf()

count_activos

,Equipo,count
0,Área/Sector,24
1,Puente grúa,15
2,Variador de frecuencia,14
3,Malacate,14
4,Motor eléctrico AC,13
5,Electroválvula,12
6,Motor eléctrico CC,11
7,Pistón hidráulico,5
8,Bomba hidráulica,5
9,Generador eléctrico,5
